In [ ]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from tqdm import tqdm

In [ ]:
# ============================================================
# DATASET PATHS
# ============================================================

MERGED_DATASET_DIR = Path(
    r"c:\Users\harsh\OneDrive\Desktop\AIML\DL\ISRO_HACKATHON\dataset\SolexCopyTest\MERGED_DATASET"
)

OUTPUT_DIR = Path(
    r"c:\Users\harsh\OneDrive\Desktop\AIML\DL\ISRO_HACKATHON\dataset\SolexCopyTest\PREPROCESSED_DATASET"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
# ============================================================
# FIND ALL MERGED FILES
# ============================================================

merged_files = sorted(
    MERGED_DATASET_DIR.glob("*.parquet")
)

print("=" * 60)
print("Merged Dataset :", MERGED_DATASET_DIR)
print("Output Folder  :", OUTPUT_DIR)
print("=" * 60)

print(f"Total merged files : {len(merged_files)}")

print("\nFirst five files:")

for file in merged_files[:5]:
    print(file.name)

In [ ]:
# ============================================================
# PARSE FLARE REGIONS
# ============================================================

def parse_flare_regions(df):
    """
    Returns

    region1 :
        List of (start_index, peak_index)

    region2 :
        List of (peak_index, end_index)
    """

    region1 = []
    region2 = []

    current_start = None
    current_peak = None

    for idx, status in df["status"].items():

        # ----------------------------------------------------
        # EVENT START
        # ----------------------------------------------------

        if status == "EVENT_START":

            # Ignore duplicate STARTs
            if current_start is None:
                current_start = idx

        # ----------------------------------------------------
        # EVENT PEAK
        # ----------------------------------------------------

        elif status == "EVENT_PEAK":

            if current_start is not None:

                current_peak = idx

                region1.append(
                    (current_start, current_peak)
                )

        # ----------------------------------------------------
        # EVENT END
        # ----------------------------------------------------

        elif status == "EVENT_END":

            if current_peak is not None:

                region2.append(
                    (current_peak, idx)
                )

                current_start = None
                current_peak = None

    return region1, region2

In [ ]:
# ============================================================
# FILL REGION 1
# ============================================================

def fill_region1(df, region1):
    """
    Fill xrsb_flux between

        EVENT_START  --->  EVENT_PEAK

    using

        weight = (LC(t)-LC(start))
                 -------------------
                 (LC(peak)-LC(start))

        xrsb(t) =
            xrsb(start)
            +
            weight *
            (xrsb(peak)-xrsb(start))
    """

    df = df.copy()

    for start_idx, peak_idx in region1:

        # ----------------------------------------------------
        # Values at START
        # ----------------------------------------------------

        lc_start = df.at[start_idx, "lc_counts_scaled"]
        lc_peak = df.at[peak_idx, "lc_counts_scaled"]

        xrsb_start = df.at[start_idx, "xrsb_flux"]
        xrsb_peak = df.at[peak_idx, "xrsb_flux"]

        # ----------------------------------------------------
        # Skip invalid region
        # ----------------------------------------------------

        if lc_peak == lc_start:
            continue

        # ----------------------------------------------------
        # Fill START -> PEAK
        # ----------------------------------------------------

        for idx in range(start_idx + 1, peak_idx):

            # Keep original GOES values
            if pd.notna(df.at[idx, "xrsb_flux"]):
                continue

            lc_t = df.at[idx, "lc_counts_scaled"]

            weight = (
                (lc_t - lc_start)
                /
                (lc_peak - lc_start)
            )

            xrsb_t = (
                xrsb_start
                +
                weight
                *
                (xrsb_peak - xrsb_start)
            )

            df.at[idx, "xrsb_flux"] = xrsb_t

    return df

In [ ]:
# ============================================================
# FILL REGION 2
# ============================================================

def fill_region2(df, region2):
    """
    Fill xrsb_flux between

        EVENT_PEAK  --->  EVENT_END

    using

        weight = (LC(peak)-LC(t))
                 -----------------
                 (LC(peak)-LC(end))

        xrsb(t) =
            xrsb(peak)
            -
            weight *
            (xrsb(peak)-xrsb(end))
    """

    df = df.copy()

    for peak_idx, end_idx in region2:

        # ----------------------------------------------------
        # Values at PEAK
        # ----------------------------------------------------

        lc_peak = df.at[peak_idx, "lc_counts_scaled"]
        lc_end = df.at[end_idx, "lc_counts_scaled"]

        xrsb_peak = df.at[peak_idx, "xrsb_flux"]
        xrsb_end = df.at[end_idx, "xrsb_flux"]

        # ----------------------------------------------------
        # Skip invalid region
        # ----------------------------------------------------

        if lc_peak == lc_end:
            continue

        # ----------------------------------------------------
        # Fill PEAK -> END
        # ----------------------------------------------------

        for idx in range(peak_idx + 1, end_idx):

            # Preserve original GOES values
            if pd.notna(df.at[idx, "xrsb_flux"]):
                continue

            lc_t = df.at[idx, "lc_counts_scaled"]

            weight = (
                (lc_peak - lc_t)
                /
                (lc_peak - lc_end)
            )

            xrsb_t = (
                xrsb_peak
                -
                weight
                *
                (xrsb_peak - xrsb_end)
            )

            df.at[idx, "xrsb_flux"] = xrsb_t

    return df

In [ ]:
# ============================================================
# PROCESS ALL MERGED FILES
# ============================================================

success = 0
failed = []

print("=" * 60)
print("Starting preprocessing...")
print("=" * 60)

for file in tqdm(merged_files):

    try:

        # ----------------------------------------------------
        # Read merged parquet
        # ----------------------------------------------------

        df = pd.read_parquet(file)

        # ----------------------------------------------------
        # Parse flare regions
        # ----------------------------------------------------

        region1, region2 = parse_flare_regions(df)

        # ----------------------------------------------------
        # Fill Region 1
        # ----------------------------------------------------

        df = fill_region1(df, region1)

        # ----------------------------------------------------
        # Fill Region 2
        # ----------------------------------------------------

        df = fill_region2(df, region2)

        # ====================================================
        # COMPUTE REGION 3 AVERAGE BACKGROUND FLUX
        # ====================================================

        background_values = []

        end_indices = df.index[df["status"] == "EVENT_END"].tolist()
        start_indices = df.index[df["status"] == "EVENT_START"].tolist()

        for end_idx in end_indices:

            next_start = next(
                (idx for idx in start_indices if idx > end_idx),
                None
            )

            if next_start is None:
                region = slice(end_idx + 1, len(df))
            else:
                region = slice(end_idx + 1, next_start)

            vals = df.loc[region, "xrsb_flux"].dropna().values

            if len(vals):
                background_values.extend(vals)

        background_flux = np.mean(background_values)
        # ----------------------------------------------------
        # Remove Region 0
        # ----------------------------------------------------

        df = (
            df
            .dropna(subset=["xrsb_flux"])
            .reset_index(drop=True)
        )

        # ----------------------------------------------------
        # Enforce Region 3 Background
        # ----------------------------------------------------

        df.loc[
            df["xrsb_flux"] < background_flux,
            "xrsb_flux"
        ] = background_flux

        # ----------------------------------------------------
        # Remove unnecessary columns
        # ----------------------------------------------------

        df = df.drop(
            columns=[
                "status",
                "flare_class"
            ]
        )

        # ----------------------------------------------------
        # Save parquet
        # ----------------------------------------------------

        output_file = OUTPUT_DIR / file.name

        df.to_parquet(
            output_file,
            index=False
        )

        success += 1

    except Exception as e:

        failed.append((file.name, str(e)))

        print(f"\nFAILED : {file.name}")
        print(e)

# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)

print(f"Successful : {success}")
print(f"Failed     : {len(failed)}")

if failed:

    print("\nFailed files:")

    for file, err in failed:
        print(f"{file} -> {err}")